# Multi-Agent RAG System — TechNova Solutions

Este notebook demuestra el sistema multi-agente RAG de TechNova Solutions.
Cubre desde la carga de documentos y vector stores hasta el enrutamiento inteligente
con el orquestador, pasando por la definicion de cada agente especializado y la
evaluacion automatizada de respuestas.

## 1. Setup e Imports

En esta seccion configuramos el entorno de trabajo: ajustamos el `sys.path` para que
los modulos del paquete `src` sean importables desde el directorio `notebooks/`,
cargamos las variables de entorno desde `.env` y verificamos que las claves necesarias
esten disponibles. Tambien mostramos las versiones de las librerias clave.

In [ ]:
import os
import sys
import json

# ── Correccion de path y directorio de trabajo ────────────────────────────────
# Los notebooks estan en notebooks/, pero src/ y data/ estan un nivel arriba.
# Cambiamos el cwd a la raiz del proyecto para que las rutas relativas
# (ej. data/hr_docs) resuelvan correctamente, e insertamos ".." al inicio
# de sys.path para que `from src.x import Y` funcione sin instalar el paquete.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    sys.path.insert(0, "..")

project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Directorio de trabajo : {os.getcwd()}")
print(f"sys.path[0]           : {sys.path[0]}")

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # carga .env desde la raiz del proyecto (directorio de trabajo actual)

# ── Verificar API keys requeridas ─────────────────────────────────────────────
openai_key = os.getenv("OPENAI_API_KEY", "")
assert openai_key, "OPENAI_API_KEY no esta configurada — agregala a tu archivo .env"
print(f"OPENAI_API_KEY      : {'*' * 8}{openai_key[-4:]}")

langfuse_pub = os.getenv("LANGFUSE_PUBLIC_KEY", "")
langfuse_sec = os.getenv("LANGFUSE_SECRET_KEY", "")
langfuse_host = os.getenv("LANGFUSE_BASE_URL", os.getenv("LANGFUSE_HOST", "https://cloud.langfuse.com"))

if langfuse_pub and langfuse_sec:
    print(f"LANGFUSE_PUBLIC_KEY : configurada ({langfuse_pub[:8]}...)")
    print(f"LANGFUSE_SECRET_KEY : configurada")
    print(f"LANGFUSE_BASE_URL   : {langfuse_host}")
else:
    print("Claves LANGFUSE no configuradas — el tracing se omitira")

In [ ]:
import importlib.metadata as meta

# ── Versiones de librerias ────────────────────────────────────────────────────
packages = ["langchain", "langfuse", "chromadb", "openai"]
for pkg in packages:
    try:
        version = meta.version(pkg)
    except meta.PackageNotFoundError:
        version = "no instalado"
    print(f"{pkg:<12} : {version}")

In [ ]:
from src.config import Settings

settings = Settings()
print(f"Modelo LLM      : {settings.MODEL_NAME}")
print(f"Tamano de chunk  : {settings.CHUNK_SIZE}")
print(f"Solapamiento     : {settings.CHUNK_OVERLAP}")

## 2. Carga de Documentos y Vector Stores

El sistema RAG necesita que los documentos de cada dominio (RRHH, Tecnologia y Finanzas)
esten indexados en colecciones vectoriales de ChromaDB.

- `DocumentLoader.load_and_split()` lee los archivos Markdown de cada dominio y los
  fragmenta en *chunks* usando `RecursiveCharacterTextSplitter`.
- `VectorStoreManager.initialize_all_stores()` crea o recarga las tres colecciones en
  `./chroma_db`. Si ya existen en disco, las reutiliza sin re-embeber.

In [ ]:
from src.document_loader import DocumentLoader
from src.vector_store import VectorStoreManager

loader = DocumentLoader()

# ── Cargar y fragmentar cada dominio ──────────────────────────────────────────
dominios = {
    "hr"     : "data/hr_docs",
    "tech"   : "data/tech_docs",
    "finance": "data/finance_docs",
}

domain_chunks = {}
for dominio, directorio in dominios.items():
    chunks = loader.load_and_split(directory=directorio, domain=dominio)
    domain_chunks[dominio] = chunks
    print(f"  → {dominio:8s}: {len(chunks):4d} chunks desde '{directorio}'")

print(f"\nTotal de chunks: {sum(len(c) for c in domain_chunks.values())}")

In [ ]:
# ── Inicializar (o recargar) vector stores ────────────────────────────────────
vsm = VectorStoreManager()
stores = vsm.initialize_all_stores()

print("\n── Estadisticas por coleccion ──")
for nombre_coleccion, store in stores.items():
    count = store._collection.count()
    print(f"  {nombre_coleccion:<15}: {count:4d} chunks")

In [ ]:
# ── Prueba rapida de retrieval sobre hr_docs ─────────────────────────────────
consulta_prueba = "What is TechNova's vacation policy?"
hr_retriever = vsm.get_retriever("hr_docs", k=3)
resultados = hr_retriever.invoke(consulta_prueba)

print(f"Consulta          : '{consulta_prueba}'")
print(f"Chunks recuperados: {len(resultados)}\n")

for i, doc in enumerate(resultados, 1):
    fuente = doc.metadata.get("source_file", "desconocido")
    preview = doc.page_content[:200].replace("\n", " ")
    print(f"[{i}] fuente: {fuente}")
    print(f"    {preview}...\n")

## 3. Definicion de Agentes RAG

Cada dominio tiene un agente especializado que encapsula un pipeline RAG:

| Agente | Dominio | Retriever |
|--------|---------|----------|
| `HRAgent` | Recursos Humanos | `hr_docs` |
| `TechAgent` | Tecnologia / IT | `tech_docs` |
| `FinanceAgent` | Finanzas | `finance_docs` |

Todos heredan de `BaseRAGAgent` y exponen el metodo `invoke(query) -> dict`
con las claves `answer`, `sources`, `agent_name` y `domain`.

In [ ]:
from src.agents import HRAgent, TechAgent, FinanceAgent
from langchain_openai import ChatOpenAI

# ── LLM compartido ───────────────────────────────────────────────────────────
llm = ChatOpenAI(
    model=settings.MODEL_NAME,
    api_key=settings.OPENAI_API_KEY,
)

# ── Retrievers ────────────────────────────────────────────────────────────────
hr_retriever      = vsm.get_retriever("hr_docs")
tech_retriever    = vsm.get_retriever("tech_docs")
finance_retriever = vsm.get_retriever("finance_docs")

# ── Instanciar agentes ────────────────────────────────────────────────────────
hr_agent      = HRAgent(hr_retriever, llm)
tech_agent    = TechAgent(tech_retriever, llm)
finance_agent = FinanceAgent(finance_retriever, llm)

print("Agentes instanciados:")
for agente in [hr_agent, tech_agent, finance_agent]:
    print(f"  {agente.agent_name} (dominio={agente.domain})")

In [ ]:
# ── Prueba del agente HR ──────────────────────────────────────────────────────
hr_result = hr_agent.invoke("What are the employee benefits at TechNova?")

print(f"Agente  : {hr_result['agent_name']} ({hr_result['domain']})")
print(f"\nRespuesta:\n{hr_result['answer']}")
print(f"\nFuentes ({len(hr_result['sources'])}):", hr_result['sources'][:3])

In [ ]:
# ── Prueba del agente Tech ────────────────────────────────────────────────────
tech_result = tech_agent.invoke("How do I set up the VPN?")

print(f"Agente  : {tech_result['agent_name']} ({tech_result['domain']})")
print(f"\nRespuesta:\n{tech_result['answer']}")
print(f"\nFuentes ({len(tech_result['sources'])}):", tech_result['sources'][:3])

In [ ]:
# ── Prueba del agente Finance ─────────────────────────────────────────────────
finance_result = finance_agent.invoke("What is the expense reimbursement process?")

print(f"Agente  : {finance_result['agent_name']} ({finance_result['domain']})")
print(f"\nRespuesta:\n{finance_result['answer']}")
print(f"\nFuentes ({len(finance_result['sources'])}):", finance_result['sources'][:3])

## 4. Orquestador y Enrutamiento Inteligente

El `Orchestrator` unifica todo el pipeline:

1. **`classify_intent`** — clasifica la consulta del usuario en uno de los dominios
   (`hr`, `tech`, `finance`, `unknown`) usando un LLM con un prompt estructurado.
2. **`route`** — clasifica y despacha la consulta al agente correcto, creando un
   trace completo en Langfuse con spans para clasificacion y recuperacion RAG.
3. **`batch_route`** — procesa una lista de consultas y calcula la precision del
   enrutamiento comparando intenciones predichas contra las esperadas.

La respuesta de `route` incluye: `query`, `intent`, `confidence`, `reasoning`,
`answer`, `sources` y `agent`.

In [ ]:
from src.agents import Orchestrator, classify_intent

# ── Instanciar orquestador ────────────────────────────────────────────────────
# El Orchestrator crea internamente su propio Settings, LLM, VectorStoreManager
# y agentes, por lo que es autocontenido.
orch = Orchestrator()

# ── Demo de classify_intent directamente ──────────────────────────────────────
consulta_clasificacion = "How do I request time off?"
clasificacion = classify_intent(consulta_clasificacion, orch.llm)

print(f"Consulta   : '{consulta_clasificacion}'")
print(f"Intencion  : {clasificacion['intent']}")
print(f"Confianza  : {clasificacion['confidence']:.2f}")
print(f"Razonamiento: {clasificacion['reasoning']}")

In [ ]:
# ── Demo completa de route ────────────────────────────────────────────────────
consulta_route = "How do I request time off?"
respuesta = orch.route(consulta_route)

print(f"Consulta      : {respuesta['query']}")
print(f"Intencion     : {respuesta['intent']}  (confianza={respuesta['confidence']:.2f})")
print(f"Razonamiento  : {respuesta['reasoning']}")
print(f"Agente usado  : {respuesta['agent']}")
print(f"\nRespuesta:\n{respuesta['answer']}")
print(f"\nFuentes ({len(respuesta['sources'])}):", respuesta['sources'][:3])

In [ ]:
# ── Demo de enrutamiento multi-dominio ────────────────────────────────────────
consultas_demo = [
    "How do I configure multi-factor authentication for my work account?",
    "What documentation do I need to submit for international travel expenses?",
]

for consulta in consultas_demo:
    resultado = orch.route(consulta)
    print("=" * 70)
    print(f"Consulta      : {resultado['query']}")
    print(f"Intencion     : {resultado['intent']}  (confianza={resultado['confidence']:.2f})")
    print(f"Agente usado  : {resultado['agent']}")
    respuesta_preview = resultado['answer'][:300].replace("\n", " ")
    print(f"Respuesta     : {respuesta_preview}{'...' if len(resultado['answer']) > 300 else ''}")
    print(f"Fuentes       : {resultado['sources'][:2]}")
    print()

## 5. Pruebas y Ejemplos — Batch Testing

En esta seccion cargamos el conjunto de consultas de prueba (`data/test_queries.json`)
y las pasamos al metodo `batch_route` del orquestador para evaluar la precision del
clasificador de intenciones a escala.

- **`batch_route(queries)`** procesa cada consulta, clasifica la intencion y la compara
  con `expected_intent` cuando esta disponible.
- El resultado incluye metricas globales (`accuracy`, `correct`, `evaluated`) y el
  detalle de cada prediccion (`intent_match`, `confidence`).
- La tabla de resultados permite identificar rapidamente los casos fallidos y su nivel
  de dificultad (easy / medium / hard).

In [ ]:
with open("data/test_queries.json") as f:
    test_data = json.load(f)
test_queries = test_data["test_queries"]
print(f"Queries de prueba cargadas: {len(test_queries)}")

# Mostrar distribucion por dificultad e intencion esperada
from collections import Counter
dist_dificultad = Counter(q["difficulty"]       for q in test_queries)
dist_intencion  = Counter(q["expected_intent"]  for q in test_queries)
print(f"Por dificultad        : {dict(dist_dificultad)}")
print(f"Por intencion esperada: {dict(dist_intencion)}")

In [ ]:
batch_results = orch.batch_route(test_queries)
print(f"Total de queries : {batch_results['total']}")
print(f"Evaluadas        : {batch_results['evaluated']}")
print(f"Correctas        : {batch_results['correct']}")
if batch_results['accuracy'] is not None:
    print(f"Precision        : {batch_results['accuracy']:.1%}")
else:
    print("Precision        : N/A")

In [ ]:
# ── Tabla de resultados ───────────────────────────────────────────────────────
print(f"{'ID':>3}  {'Consulta':<52}  {'Esperado':<9}  {'Actual':<9}  {'Conf':>5}  {'OK?'}")
print("-" * 96)

for r in batch_results["results"]:
    q_id    = r.get("id", "-")
    query   = (r["query"][:49] + "…") if len(r["query"]) > 50 else r["query"]
    esperado = r.get("expected_intent", "N/A")
    actual   = r["intent"]
    conf     = r["confidence"]
    ok       = r.get("intent_match")
    marca    = "✓" if ok else ("✗" if ok is False else "—")
    print(f"{str(q_id):>3}  {query:<52}  {esperado:<9}  {actual:<9}  {conf:>5.2f}  {marca}")

print("-" * 96)
if batch_results['accuracy'] is not None:
    print(f"Precision global: {batch_results['accuracy']:.1%}  "
          f"({batch_results['correct']}/{batch_results['evaluated']} evaluadas)")

# ── Precision por dificultad ──────────────────────────────────────────────────
diff_correctas = Counter()
diff_total     = Counter()
for r in batch_results["results"]:
    diff = r.get("difficulty", "desconocida")
    diff_total[diff] += 1
    if r.get("intent_match"):
        diff_correctas[diff] += 1

print("\nPrecision por dificultad:")
for diff in ["easy", "medium", "hard"]:
    total = diff_total.get(diff, 0)
    correctas = diff_correctas.get(diff, 0)
    if total:
        print(f"  {diff:<6}: {correctas}/{total}  ({correctas/total:.1%})")
    else:
        print(f"  {diff:<6}: sin muestras")

## 6. Integracion con Langfuse — Observabilidad

Langfuse es la capa de observabilidad del sistema. Cada llamada a `orch.route()` genera
automaticamente un **trace** con spans anidados:

| Span | Descripcion |
|------|-------------|
| `user-query` | Span padre — cubre toda la consulta |
| `intent-classification` | Llamada al LLM para clasificar la intencion |
| `rag-retrieval` | Recuperacion de chunks y generacion de la respuesta |

En el dashboard de Langfuse podes explorar:
- **Latencia** por span y por agente
- **Tokens usados** (prompt + completion) en cada llamada al LLM
- **Inputs / Outputs** de cada span para depurar el razonamiento
- **Precision** del clasificador usando los `expected_intent` del batch

A continuacion verificamos la conexion y ejecutamos una consulta trazada de ejemplo.

In [ ]:
from src.tracing import get_langfuse_client

langfuse_client = get_langfuse_client()
try:
    auth_ok = langfuse_client.auth_check()
    print(f"Verificacion Langfuse: {'✓ Conectado' if auth_ok else '✗ Fallo'}")
    print(f"Host: {settings.LANGFUSE_HOST}")
except Exception as e:
    print(f"Error de conexion con Langfuse: {e}")

In [ ]:
# ── Ejecutar una consulta con tracing ─────────────────────────────────────────
consulta_trazada = "What benefits does TechNova offer to remote employees?"
print(f"Ejecutando consulta trazada: '{consulta_trazada}'")
resultado_trazado = orch.route(consulta_trazada, user_id="notebook-demo")

print(f"\nIntencion  : {resultado_trazado['intent']} (confianza={resultado_trazado['confidence']:.2f})")
print(f"Agente     : {resultado_trazado['agent']}")
print(f"Respuesta  : {resultado_trazado['answer'][:300]}...")
print(f"\nFuentes    : {resultado_trazado['sources'][:3]}")
print(f"\n→ Revisa tu dashboard de Langfuse en {settings.LANGFUSE_HOST}")
print("  Busca el trace 'user-query' con spans: intent-classification, rag-retrieval")

### Lo que Langfuse captura por cada llamada a `orch.route()`

```
Trace: user-query
  ├── Span: intent-classification
  │     input : consulta original del usuario
  │     output: {intent, confidence, reasoning}
  │     modelo: gpt-4o-mini (o el MODEL_NAME configurado)
  │     tokens: prompt_tokens + completion_tokens
  │
  └── Span: rag-retrieval
        input : consulta original
        output: {answer, sources, agent_name}
        modelo: mismo LLM
        chunks: documentos recuperados por el retriever
```

Para explorar los traces:
1. Abri el dashboard en `settings.LANGFUSE_HOST`
2. Navega a **Tracing → Traces**
3. Filtra por nombre `user-query` o por `user_id="notebook-demo"`
4. Hace click en cualquier trace para ver el arbol de spans, latencias y tokens

> **Tip:** Podes agregar `score` a un trace usando `langfuse_client.score()` para
> registrar metricas de calidad (ej. relevancia de la respuesta) directamente desde
> el notebook. El `ResponseEvaluator` de `evaluator.py` automatiza este proceso.